# 00 — Panduan, Setup, dan Peta Project

Notebook ini adalah **titik masuk** untuk memahami project pemodelan deteksi
alergen. Tidak ada training di sini. Tujuannya agar Anda tahu:

- apa yang dimaksud dengan istilah-istilah teknis,
- notebook mana yang harus dijalankan lebih dulu,
- berapa lama waktu yang dibutuhkan,
- dan aturan eksperimen yang tidak boleh dilanggar.

## Untuk siapa notebook ini?

Notebook ini ditulis untuk orang yang **belum tentu berlatar belakang machine
learning**. Setiap langkah penting dijelaskan dengan bahasa sehari-hari.
Kode tetap ada, tetapi logika berat disembunyikan di folder `app/experiment/`
agar notebook mudah dibaca.

## Peta notebook

| Notebook | Isi | Untuk siapa | Estimasi waktu |
|---|---|---|---|
| `00_panduan_dan_setup.ipynb` | Istilah, setup, peta project | semua | 10 menit |
| `01_data_dan_split.ipynb` | Baca data, audit, split anti-bocor, data sintetik | semua | 10–20 menit |
| `02_bilstm_dasar.ipynb` | Baseline Word2Vec + BiLSTM | pemula | 10–30 menit |
| `03_bilstm_tuning.ipynb` | Hyperparameter tuning BiLSTM lengkap | peneliti | 1–6 jam |
| `04_indobert_dasar.ipynb` | Baseline fine-tuning IndoBERT | pemula | 10–40 menit (GPU) |
| `05_indobert_tuning.ipynb` | Hyperparameter tuning IndoBERT lengkap | peneliti | 2–12 jam (GPU) |
| `06_perbandingan_dan_laporan.ipynb` | Bandingkan model, uji statistik, ekspor laporan | peneliti | 10 menit |

## Istilah penting

| Istilah | Arti sederhana |
|---|---|
| **Komposisi** | Daftar bahan pada kemasan makanan. |
| **OCR** | Proses mengubah gambar tulisan menjadi teks. |
| **Gold label** | Label acuan yang dianggap benar (dibuat manusia). |
| **Knowledge Base (KB)** | Aturan kata kunci alergen, dipakai sebagai pembanding/diagnostik. KB bukan pengganti gold label. |
| **Leakage** | Kebocoran informasi: data ujian ikut masuk ke data latihan. Ini membuat skor terlihat bagus padahal tidak jujur. |
| **Split** | Pembagian data menjadi train (latihan), validation (pemilihan model), dan holdout (ujian akhir). |
| **Frozen holdout** | Holdout yang sudah dikunci dan tidak boleh diubah-ubah. |
| **Word2Vec** | Mengubah kata menjadi vektor angka yang menyimpan makna. |
| **BiLSTM** | Model urutan dua arah yang membaca konteks kata dari kiri dan kanan. |
| **IndoBERT** | Model bahasa Transformer pra-latih untuk Bahasa Indonesia. |
| **Hyperparameter** | Pengaturan model yang dipilih sebelum training, mis. learning rate, jumlah unit, dropout. |
| **HPO** | Hyperparameter Optimization: mencari kombinasi hyperparameter terbaik secara sistematis. |
| **ROC-AUC** | Kemampuan model membedakan dua kelas; 0,5 = acak, 1,0 = sempurna. |
| **F1** | Keseimbangan precision dan recall. |
| **Bootstrap CI** | Rentang ketidakpastian metrik, dihitung dengan resampling. |

## Cara menjalankan

> Jalankan notebook dari sel pertama (`Restart Kernel and Run All Cells`).
> Notebook memakai state antar sel; melompat ke sel tengah bisa memunculkan
> `NameError`.

Disarankan memakai **Python 3.10–3.12**. TensorFlow 2.16 belum mendukung
Python 3.14, jadi environment yang dipakai untuk training harus Python 3.12
(atau 3.10/3.11).

```bash
# dari root project
python3.12 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\activate
pip install -r requirements-train.txt
pip install -r requirements-notebook.txt

# agar hash Python deterministik, set SEBELUM python berjalan
export PYTHONHASHSEED=42           # Windows PowerShell: $env:PYTHONHASHSEED=42

jupyter lab
```

Untuk IndoBERT, install stack terpisah:

```bash
pip install -r requirements-bert.txt
```

Untuk strategi HPO Bayesian (opsional):

```bash
pip install -r requirements-hpo.txt
```

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebook biasanya dijalankan dari folder notebooks/. Kita pindah ke root
# project agar semua path artefak konsisten.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "artifacts" / "data"
BERT_DIR = PROJECT_ROOT / "artifacts" / "bert" / "input"
RUNS_DIR = PROJECT_ROOT / "artifacts" / "notebook_runs"
for path in (DATA_DIR, BERT_DIR, RUNS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Data dir     :", DATA_DIR)
print("BERT input   :", BERT_DIR)
print("Runs dir     :", RUNS_DIR)

## Cek environment

Sel berikut menampilkan versi paket yang terpasang. Jika TensorFlow/Gensim
belum ada, notebook data tetap bisa jalan, tetapi notebook training BiLSTM
akan memberi pesan yang jelas.

In [ ]:
from app.experiment.artifacts import environment_snapshot

snapshot = environment_snapshot(seed=42)
print("Python:", sys.version.split()[0])
print("Git SHA:", snapshot["git_sha"])
print("GPU:", snapshot["gpu"])
print("\nPaket penting:")
for name, version in snapshot["packages"].items():
    print(f"  {name:15s} {version}")

if snapshot["reproducibility_warnings"]:
    print("\nPeringatan reproducibility:")
    for warning in snapshot["reproducibility_warnings"]:
        print("  -", warning)

## Cari file dataset

Dataset yang dipakai di project ini memiliki kolom:

- `nama produk` — identitas produk,
- `text` — teks komposisi hasil OCR,
- `label` — `safe` atau `unsafe`.

Sel berikut mencari file CSV di lokasi yang umum.

In [ ]:
candidates = [
    PROJECT_ROOT / "ocr_output" / "data-mengandung.csv",
    PROJECT_ROOT.parent / "old" / "ocr_output" / "data-mengandung.csv",
    PROJECT_ROOT / "data.csv",
]
for path in candidates:
    print(("[ADA]    " if path.exists() else "[TIDAK]  ") + str(path))

## Aturan main eksperimen (penting untuk paper)

1. **Holdout tidak boleh disentuh** untuk training, early stopping, atau
   memilih hyperparameter. Holdout hanya diprediksi sekali di akhir.
2. **Validation boleh dipakai untuk memilih model.** Itulah tugasnya.
3. **Threshold utama tetap 0,5.** Jangan mencari threshold yang paling bagus
   di holdout, karena itu termasuk tuning di data ujian.
4. **Setiap angka harus menyebut n dan interval kepercayaan.**
5. **Klaim "model A lebih baik" harus memakai uji berpasangan** (McNemar /
   paired bootstrap), bukan hanya selisih angka.
6. **Jalankan beberapa seed** untuk HPO agar hasil tidak bergantung pada satu
   undian acak.

## Estimasi waktu dan kebutuhan

| Kegiatan | CPU | GPU |
|---|---|---|
| Notebook 01 (data + split) | < 1 menit | < 1 menit |
| Baseline BiLSTM 20 epoch | 5–20 menit | 1–3 menit |
| HPO BiLSTM `smoke` (4 trial) | 15–60 menit | 5–15 menit |
| HPO BiLSTM `standard` (30 trial) | 2–8 jam | 30–90 menit |
| Baseline IndoBERT 4 epoch | 1–3 jam | 5–15 menit |
| HPO IndoBERT `smoke` (4 trial) | 4–12 jam | 20–60 menit |

Waktu sangat bergantung pada CPU/GPU, panjang teks, dan batch size.

## Troubleshooting singkat

| Gejala | Penyebab umum | Solusi |
|---|---|---|
| `ModuleNotFoundError: tensorflow` | Environment salah | Pakai Python 3.12 + `pip install -r requirements-train.txt` |
| `ModuleNotFoundError: gensim` | Belum install | `pip install gensim==4.3.3` |
| Training sangat lambat | CPU / max_len besar | Turunkan `epochs`, `max_len`, atau pakai GPU |
| `CUDA out of memory` | Batch terlalu besar | Turunkan `batch_size` atau aktifkan `gradient_accumulation_steps` |
| Hasil berbeda setiap run | Hash randomization / seed | Set `PYTHONHASHSEED=42` sebelum Python, pakai seed yang sama |
| `Deterministic GPU implementation ... not available` | Metric Keras/ops tertentu tidak deterministik di GPU | Versi ini sudah memakai metric custom; jika masih muncul, set `config.deterministic = False` |
| `FileNotFoundError` CSV | Path salah | Set `DataConfig.csv_input` ke path absolut |
| Model BERT tidak download | Tidak ada internet | Unduh di login node, set `HF_HUB_OFFLINE=1` |